# tsfresh feature generation and t-sne analysis on StressID and ExpData datasets

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from dataclasses import dataclass
from typing import Tuple, TypeAlias

from tsfresh import extract_features, select_features

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer

BASE_PATH = "../../.."
DATASET = f"{BASE_PATH}/stressid-dataset"
DATA_ECG = f"{DATASET}/ecg_windowed.csv"
DATA_EDA = f"{DATASET}/eda_windowed.csv"
LABELS_SEPARATOR = ","
LABELS = f"{BASE_PATH}/stressID/labels.csv"
DATA_SEPARATOR = ","
DATA_FS = 500  # Hz
DATA_WINDOW_DURATION = 60  # seconds
TARGET_FS = 51.2
RANDOM_STATE = 21

BIN_LABELS = ["NoStress", "Stress"]
TER_LABELS = ["Relaxed", "Stress", "RealStress"]
QAD_LABELS = ["Relaxed", "Stress", "RealStress", "Amused"]

LABELS_CONF = {
    "b": {
        "col_name": "binary-stress",
        "enabled": True,
        "stratification": True,
        "classes": BIN_LABELS,
    },
    "t": {
        "col_name": "affect3-class",
        "enabled": False,
        "stratification": True,
        "classes": TER_LABELS,
    },
    "q": {
        "col_name": "affect4-class",
        "enabled": False,
        "stratification": True,
        "classes": QAD_LABELS,
    },
}


@dataclass
class Dataset:
    X: list[pd.Series]
    y: pd.Series
    groups: np.ndarray[int]


CWT: TypeAlias = Tuple[np.ndarray[tuple[int], np.dtype], np.ndarray]

### Build dataset from data files

In [2]:
# Creating labels object

labels_df = pd.read_csv(LABELS, sep=LABELS_SEPARATOR, header=0, index_col=0)
labels: dict[str, pd.Series] = {}
for key, conf in LABELS_CONF.items():
    if conf["enabled"]:
        labels[key] = labels_df[conf["col_name"]]

display(labels["b"])


subject/task
2ea4_Breathing    0
2ea4_Counting1    1
2ea4_Counting2    1
2ea4_Counting3    1
2ea4_Math         1
                 ..
y9z6_Relax        0
y9z6_Speaking     1
y9z6_Stroop       1
y9z6_Video1       1
y9z6_Video2       0
Name: binary-stress, Length: 700, dtype: int64

In [ ]:
# Pairing labels and samples for each class type

raw_num_samples = DATA_FS * DATA_WINDOW_DURATION
raw_df = pd.read_csv(DATA_EDA)

datasets: dict[str, Dataset] = {}
for class_type, labels_set in labels.items():
    subject_to_group: dict[str, int] = {}
    group_counter = 0
    samples: list[pd.Series] = []
    groups_list: list[int] = []
    dataset_labels = pd.Series([], dtype=np.int64, name=labels_set.name)
    for col_name, label in labels_set.items():
        subject_id = col_name.split("_")[0]
        if (
            col_name in raw_df.columns
        ):  # Only add labels with corresponding data, EDA dataset lacks one entry which ECG has
            samples.append(raw_df[col_name])
            dataset_labels[col_name] = label
            if subject_id not in subject_to_group:
                subject_to_group[subject_id] = group_counter
                group_counter += 1
            groups_list.append(subject_to_group[subject_id])
    groups = np.array(groups_list)
    datasets[class_type] = Dataset(samples, dataset_labels, groups)
    print(
        f"Class type: {class_type}. Saving {len(datasets[class_type].X)} samples with {len(datasets[class_type].y)} labels"
    )
    print(f"{len(subject_to_group)} groups found. Indexed {len(groups)}.")


In [ ]:
#display(datasets['b'].groups)
display(datasets["b"].X)
display(datasets["b"].y)